# Power Analysis and Sample Size Calculation for A/B Tests

This notebook demonstrates how to:
1. Calculate required sample sizes for desired power
2. Understand the relationship between effect size, sample size, and power
3. Plan experiments with clustered data (multiple sessions per user)
4. Account for covariate adjustment in sample size calculations

## Key Concepts

- **Power**: Probability of detecting a true effect (typically 80%)
- **Type I Error (α)**: False positive rate (typically 5%)
- **MDE**: Minimum Detectable Effect - smallest effect we want to detect
- **Design Effect**: Inflation factor due to clustering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import brentq
from ab_glm import simulate_ab_data, fit_binomial_glm, marginal_effects_ate_and_rr

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Basic Sample Size Calculation (No Clustering)

In [ ]:
def sample_size_binary(p_control, mde, alpha=0.05, power=0.80, ratio=1):
    """
    Calculate sample size for binary outcome A/B test.
    
    Parameters:
    -----------
    p_control : float
        Expected conversion rate in control group
    mde : float
        Minimum detectable effect (absolute difference)
    alpha : float
        Significance level (Type I error rate)
    power : float
        Statistical power (1 - Type II error rate)
    ratio : float
        Ratio of treatment to control sample size
    
    Returns:
    --------
    n_control, n_treatment : tuple
        Required sample sizes for each group
    """
    p_treatment = p_control + mde
    p_pooled = (p_control + ratio * p_treatment) / (1 + ratio)
    
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    
    # Standard formula for two-sample proportion test
    n_control = (
        (z_alpha * np.sqrt((1 + ratio) * p_pooled * (1 - p_pooled)) +
         z_beta * np.sqrt(p_control * (1 - p_control) + 
                         ratio * p_treatment * (1 - p_treatment))) ** 2 /
        (mde ** 2 * ratio)
    )
    
    n_treatment = n_control * ratio
    
    return int(np.ceil(n_control)), int(np.ceil(n_treatment))

# Example calculation
p_baseline = 0.10  # 10% baseline conversion
mde = 0.02  # 2 percentage point lift
alpha = 0.05
power = 0.80

n_control, n_treatment = sample_size_binary(p_baseline, mde, alpha, power)

print("="*60)
print("BASIC SAMPLE SIZE CALCULATION")
print("="*60)
print(f"Baseline conversion rate: {p_baseline:.1%}")
print(f"Minimum detectable effect: {mde:.1%} (absolute)")
print(f"Target conversion rate: {(p_baseline + mde):.1%}")
print(f"Relative lift: {(mde/p_baseline)*100:.1f}%")
print(f"\nSignificance level (α): {alpha}")
print(f"Statistical power: {power:.0%}")
print(f"\nRequired sample size:")
print(f"  Control group: {n_control:,} users")
print(f"  Treatment group: {n_treatment:,} users")
print(f"  Total: {n_control + n_treatment:,} users")

## 2. Sample Size vs Effect Size Relationship

In [ ]:
# Explore how sample size changes with MDE
mde_range = np.linspace(0.005, 0.05, 50)
baseline_rates = [0.05, 0.10, 0.20, 0.30]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute MDE
for p_base in baseline_rates:
    sample_sizes = [sample_size_binary(p_base, mde, alpha, power)[0] * 2 
                   for mde in mde_range]
    axes[0].plot(mde_range * 100, sample_sizes, 
                label=f'Baseline: {p_base:.0%}', linewidth=2)

axes[0].set_xlabel('Minimum Detectable Effect (percentage points)')
axes[0].set_ylabel('Total Sample Size Required')
axes[0].set_title('Sample Size vs Absolute Effect Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')
axes[0].set_ylim([100, 1000000])

# Relative MDE
relative_lifts = np.linspace(0.05, 0.50, 50)  # 5% to 50% relative lift
for p_base in baseline_rates:
    sample_sizes = [sample_size_binary(p_base, p_base * lift, alpha, power)[0] * 2 
                   for lift in relative_lifts]
    axes[1].plot(relative_lifts * 100, sample_sizes,
                label=f'Baseline: {p_base:.0%}', linewidth=2)

axes[1].set_xlabel('Relative Lift (%)')
axes[1].set_ylabel('Total Sample Size Required')
axes[1].set_title('Sample Size vs Relative Effect Size')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')
axes[1].set_ylim([100, 1000000])

plt.tight_layout()
plt.show()

print("Key insights:")
print("• Sample size increases dramatically for smaller effects")
print("• Lower baseline rates require larger samples for same absolute effect")
print("• Higher baseline rates require larger samples for same relative effect")

## 3. Power Curves

In [ ]:
def calculate_power(n, p_control, effect_size, alpha=0.05):
    """
    Calculate statistical power for given sample size and effect.
    """
    p_treatment = p_control + effect_size
    p_pooled = (p_control + p_treatment) / 2
    
    se_null = np.sqrt(2 * p_pooled * (1 - p_pooled) / n)
    se_alt = np.sqrt((p_control * (1 - p_control) + 
                     p_treatment * (1 - p_treatment)) / n)
    
    z_alpha = stats.norm.ppf(1 - alpha/2)
    critical_value = z_alpha * se_null
    
    z_power = (effect_size - critical_value) / se_alt
    power = stats.norm.cdf(z_power)
    
    return power

# Create power curves
sample_sizes = [500, 1000, 2000, 5000, 10000]
effect_sizes = np.linspace(0, 0.05, 100)
p_baseline = 0.10

plt.figure(figsize=(10, 6))

for n in sample_sizes:
    powers = [calculate_power(n, p_baseline, es, alpha) for es in effect_sizes]
    plt.plot(effect_sizes * 100, powers, label=f'n = {n:,} per group', linewidth=2)

plt.axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='80% power')
plt.xlabel('Effect Size (percentage points)')
plt.ylabel('Statistical Power')
plt.title(f'Power Curves for Different Sample Sizes (Baseline: {p_baseline:.0%})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.show()

# Find MDE for each sample size at 80% power
print("\nMinimum Detectable Effect at 80% Power:")
print("="*40)
for n in sample_sizes:
    # Find MDE using root finding
    def power_diff(es):
        return calculate_power(n, p_baseline, es, alpha) - 0.80
    
    if power_diff(0.001) < 0 and power_diff(0.10) > 0:
        mde = brentq(power_diff, 0.001, 0.10)
        print(f"n = {n:>6,} per group: MDE = {mde*100:>5.2f}% ({mde/p_baseline*100:.1f}% relative)")

## 4. Accounting for Clustering (Multiple Sessions per User)

In [ ]:
def design_effect_clustering(sessions_per_user, icc):
    """
    Calculate design effect for clustered data.
    
    Parameters:
    -----------
    sessions_per_user : float
        Average number of sessions per user
    icc : float
        Intra-cluster correlation coefficient
    
    Returns:
    --------
    deff : float
        Design effect multiplier
    """
    return 1 + (sessions_per_user - 1) * icc

# Estimate ICC from simulated data
df = simulate_ab_data(n_users=1000, sessions_per_user=(2, 6), seed=42)

# Calculate ICC using variance components
from scipy.stats import f_oneway

# Group data by user
groups = [group['y'].values for _, group in df.groupby('user_id')]
# Simple estimate of ICC
total_var = df['y'].var()
within_var = np.mean([g.var() for g in groups if len(g) > 1])
between_var = total_var - within_var
icc_estimate = between_var / total_var if total_var > 0 else 0

avg_sessions = df.groupby('user_id').size().mean()

print("="*60)
print("CLUSTERING EFFECTS ON SAMPLE SIZE")
print("="*60)
print(f"Average sessions per user: {avg_sessions:.2f}")
print(f"Estimated ICC: {icc_estimate:.4f}")
print(f"Design effect: {design_effect_clustering(avg_sessions, icc_estimate):.2f}")
print()

# Compare sample sizes with and without clustering
p_baseline = 0.10
mde = 0.02

n_simple = sample_size_binary(p_baseline, mde, alpha, power)[0]
deff = design_effect_clustering(avg_sessions, icc_estimate)
n_clustered = int(np.ceil(n_simple * deff))

print("Sample size requirements:")
print(f"  Without clustering: {n_simple:,} users per group")
print(f"  With clustering: {n_clustered:,} users per group")
print(f"  Inflation factor: {deff:.2f}x")
print(f"\nTotal observations:")
print(f"  {n_clustered:,} users × {avg_sessions:.1f} sessions = {int(n_clustered * avg_sessions):,} observations per group")

## 5. Covariate Adjustment Benefits

In [ ]:
def variance_reduction_factor(r_squared):
    """
    Calculate variance reduction from covariate adjustment.
    
    Parameters:
    -----------
    r_squared : float
        R-squared from regressing outcome on covariates
    
    Returns:
    --------
    factor : float
        Variance reduction factor (< 1 means reduction)
    """
    return 1 - r_squared

# Simulate to estimate R-squared from covariates
df_large = simulate_ab_data(n_users=5000, sessions_per_user=(3, 5), seed=42)

# Fit model with just covariates (no treatment)
from sklearn.linear_model import LogisticRegression
X = df_large[['country_EU', 'device_mobile', 'prior_views']].values
y = df_large['y'].values

# Use logistic regression to estimate variance explained
lr = LogisticRegression(random_state=42)
lr.fit(X, y)

# McFadden's pseudo R-squared
from sklearn.metrics import log_loss
null_prob = y.mean()
null_ll = -log_loss(y, [null_prob] * len(y), normalize=False)
model_ll = -log_loss(y, lr.predict_proba(X), normalize=False)
mcfadden_r2 = 1 - (model_ll / null_ll)

# Conservative estimate of variance reduction
var_reduction = variance_reduction_factor(mcfadden_r2 * 0.5)  # Conservative

print("="*60)
print("BENEFITS OF COVARIATE ADJUSTMENT")
print("="*60)
print(f"McFadden's pseudo R²: {mcfadden_r2:.4f}")
print(f"Conservative variance reduction: {(1-var_reduction)*100:.1f}%")
print(f"Effective sample size multiplier: {1/var_reduction:.2f}x")
print()

# Compare required sample sizes
n_unadjusted = sample_size_binary(p_baseline, mde, alpha, power)[0]
n_adjusted = int(np.ceil(n_unadjusted * var_reduction))

print("Sample size requirements (per group):")
print(f"  Without covariates: {n_unadjusted:,} users")
print(f"  With covariates: {n_adjusted:,} users")
print(f"  Reduction: {n_unadjusted - n_adjusted:,} users ({(1-var_reduction)*100:.1f}%)")
print()
print("Note: Actual reduction depends on covariate quality and their")
print("correlation with the outcome. Always validate with simulations.")

## 6. Simulation-Based Power Analysis

In [ ]:
def simulate_power_glm(n_users, true_effect, n_simulations=100, 
                      sessions_per_user=(2, 5), alpha=0.05):
    """
    Estimate power using simulation with GLM.
    """
    significant_results = 0
    
    for sim in range(n_simulations):
        # Generate data with known effect
        df = simulate_ab_data(n_users=n_users, 
                            sessions_per_user=sessions_per_user,
                            seed=sim)
        
        # Modify to have specific treatment effect
        if true_effect != 0:
            # Add treatment effect
            treatment_mask = df['T'] == 1
            n_treatment = treatment_mask.sum()
            # Flip some outcomes to create desired effect
            n_to_flip = int(n_treatment * abs(true_effect))
            if true_effect > 0:
                # Increase conversion in treatment
                flip_candidates = df[treatment_mask & (df['y'] == 0)].index
                if len(flip_candidates) >= n_to_flip:
                    flip_idx = np.random.choice(flip_candidates, n_to_flip, replace=False)
                    df.loc[flip_idx, 'y'] = 1
        
        # Fit model
        try:
            _, _, df_model, results = fit_binomial_glm(df, link="logit")
            
            # Extract p-value for treatment effect
            treat_idx = list(results.params.index).index('T')
            treat_coef = results.params['T']
            treat_se = np.sqrt(np.diag(results.cov_params()))[treat_idx]
            z_stat = treat_coef / treat_se
            p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
            
            if p_value < alpha:
                significant_results += 1
        except:
            # Model failed to converge
            pass
    
    return significant_results / n_simulations

# Run power simulation
print("="*60)
print("SIMULATION-BASED POWER ANALYSIS")
print("="*60)
print("Running simulations (this may take a minute)...\n")

sample_sizes = [500, 1000, 2000]
true_effect = 0.02  # 2% absolute effect

for n in sample_sizes:
    power = simulate_power_glm(n, true_effect, n_simulations=50)
    print(f"n = {n:>4,} users per group: Power = {power:.1%}")

print("\nNote: Simulation accounts for:")
print("  • Clustering (multiple sessions per user)")
print("  • Covariate adjustment")
print("  • Cluster-robust standard errors")
print("  • Actual GLM fitting process")

## 7. Sample Size Calculator Tool

In [ ]:
def ab_test_sample_size_calculator(
    baseline_rate,
    minimum_detectable_effect,
    effect_type='absolute',
    alpha=0.05,
    power=0.80,
    sessions_per_user=1,
    icc=0,
    covariate_r2=0
):
    """
    Comprehensive sample size calculator for A/B tests.
    
    Parameters:
    -----------
    baseline_rate : float
        Control group conversion rate
    minimum_detectable_effect : float
        MDE (absolute or relative based on effect_type)
    effect_type : str
        'absolute' or 'relative'
    alpha : float
        Significance level
    power : float
        Statistical power
    sessions_per_user : float
        Average sessions per user (for clustering)
    icc : float
        Intra-cluster correlation
    covariate_r2 : float
        Expected R-squared from covariates
    
    Returns:
    --------
    dict : Sample size details
    """
    # Convert relative to absolute if needed
    if effect_type == 'relative':
        mde_absolute = baseline_rate * minimum_detectable_effect
    else:
        mde_absolute = minimum_detectable_effect
    
    # Basic sample size
    n_basic = sample_size_binary(baseline_rate, mde_absolute, alpha, power)[0]
    
    # Adjust for clustering
    deff = design_effect_clustering(sessions_per_user, icc)
    n_clustered = n_basic * deff
    
    # Adjust for covariates
    var_reduction = variance_reduction_factor(covariate_r2)
    n_final = int(np.ceil(n_clustered * var_reduction))
    
    return {
        'users_per_group': n_final,
        'total_users': n_final * 2,
        'observations_per_group': int(n_final * sessions_per_user),
        'total_observations': int(n_final * sessions_per_user * 2),
        'basic_sample_size': n_basic,
        'design_effect': deff,
        'variance_reduction': var_reduction,
        'absolute_mde': mde_absolute,
        'relative_mde': mde_absolute / baseline_rate
    }

# Interactive calculator
print("="*60)
print("A/B TEST SAMPLE SIZE CALCULATOR")
print("="*60)
print()

# Example scenarios
scenarios = [
    {
        'name': 'Simple Web Conversion',
        'baseline_rate': 0.05,
        'minimum_detectable_effect': 0.01,
        'effect_type': 'absolute',
        'sessions_per_user': 1,
        'icc': 0,
        'covariate_r2': 0
    },
    {
        'name': 'E-commerce with Sessions',
        'baseline_rate': 0.10,
        'minimum_detectable_effect': 0.10,  # 10% relative
        'effect_type': 'relative',
        'sessions_per_user': 3,
        'icc': 0.2,
        'covariate_r2': 0
    },
    {
        'name': 'Optimized with Covariates',
        'baseline_rate': 0.10,
        'minimum_detectable_effect': 0.10,  # 10% relative
        'effect_type': 'relative',
        'sessions_per_user': 3,
        'icc': 0.2,
        'covariate_r2': 0.15
    }
]

for scenario in scenarios:
    print(f"\nScenario: {scenario['name']}")
    print("-" * 40)
    
    result = ab_test_sample_size_calculator(**{k: v for k, v in scenario.items() if k != 'name'})
    
    print(f"Baseline rate: {scenario['baseline_rate']:.1%}")
    if scenario['effect_type'] == 'relative':
        print(f"Target effect: {scenario['minimum_detectable_effect']*100:.1f}% relative lift")
    else:
        print(f"Target effect: {scenario['minimum_detectable_effect']*100:.1f} pp absolute")
    
    print(f"\nRequired sample size:")
    print(f"  {result['users_per_group']:,} users per group")
    print(f"  {result['total_users']:,} total users")
    
    if scenario['sessions_per_user'] > 1:
        print(f"\nWith {scenario['sessions_per_user']:.1f} sessions per user:")
        print(f"  {result['observations_per_group']:,} observations per group")
        print(f"  Design effect: {result['design_effect']:.2f}x")
    
    if scenario['covariate_r2'] > 0:
        print(f"\nCovariate adjustment benefit:")
        print(f"  Variance reduction: {(1-result['variance_reduction'])*100:.1f}%")

## 8. Duration Calculator

In [ ]:
def test_duration_calculator(required_users, daily_traffic, allocation_rate=0.5):
    """
    Calculate test duration based on traffic.
    
    Parameters:
    -----------
    required_users : int
        Total users needed
    daily_traffic : int
        Daily unique users
    allocation_rate : float
        Fraction of traffic in test (safety margin)
    
    Returns:
    --------
    dict : Duration details
    """
    effective_daily = daily_traffic * allocation_rate
    days = np.ceil(required_users / effective_daily)
    weeks = days / 7
    
    return {
        'days': int(days),
        'weeks': weeks,
        'daily_users_in_test': int(effective_daily),
        'total_users_in_test': int(days * effective_daily)
    }

# Example duration calculation
print("="*60)
print("TEST DURATION CALCULATOR")
print("="*60)
print()

daily_traffic_scenarios = [1000, 5000, 10000, 50000]
required_users = 10000  # From earlier calculation

for daily in daily_traffic_scenarios:
    duration_50 = test_duration_calculator(required_users, daily, 0.5)
    duration_100 = test_duration_calculator(required_users, daily, 1.0)
    
    print(f"Daily traffic: {daily:,} users")
    print(f"  50% allocation: {duration_50['days']} days ({duration_50['weeks']:.1f} weeks)")
    print(f"  100% allocation: {duration_100['days']} days ({duration_100['weeks']:.1f} weeks)")
    print()

print("\nConsiderations for test duration:")
print("• Minimum 1-2 weeks to capture weekly patterns")
print("• Maximum 4-6 weeks to avoid seasonality issues")
print("• Consider holidays and special events")
print("• Account for ramp-up period if not starting at 100%")

## Summary and Best Practices

### Key Takeaways

1. **Sample size depends on**:
   - Baseline conversion rate
   - Minimum detectable effect (MDE)
   - Significance level (α) and power
   - Clustering (sessions per user)
   - Covariate quality

2. **Clustering increases required sample size**:
   - Design effect = 1 + (m-1)×ICC
   - Can easily require 2-3x more users

3. **Covariate adjustment reduces required sample size**:
   - Good covariates can reduce sample needs by 20-40%
   - Must be pre-treatment characteristics

4. **Practical recommendations**:
   - Use 80% power as standard (90% for critical decisions)
   - Consider both statistical and practical significance
   - Run power analysis before starting the test
   - Use simulation for complex designs

### Sample Size Quick Reference

For 80% power, α=0.05, 1:1 allocation:

| Baseline | Absolute MDE | Relative MDE | Sample Size (per group) |
|----------|--------------|--------------|------------------------|
| 5%       | 1 pp         | 20%          | ~3,800                 |
| 10%      | 1 pp         | 10%          | ~7,300                 |
| 10%      | 2 pp         | 20%          | ~1,900                 |
| 20%      | 2 pp         | 10%          | ~3,100                 |
| 30%      | 3 pp         | 10%          | ~2,900                 |

*Note: Multiply by design effect for clustered data*